## Structural metrics:

- **In/out-degree**: Counted from the raw triples directly. A node touched by 2 relations to the same neighbor counts twice. This is the true KG notion of degree.
- **Edges, triangles, clustering coefficient**: Computed on a simple undirected projection `U`. One edge per entity pair with *any* connecting triple, in either direction, across any relation.
- **Node triangles**: Standard triangle count on `U`. Each closed 3-node triad counted once, regardless of how many relations/directions link its sides.
- **Edge triangles**: Multiplicity-weighted. For each triad `{u,v,w}` in `U`, multiply the raw triple-count connecting each of its 3 sides (`mult(u,v) * mult(v,w) * mult(u,w)`) and sum over all triads. `>= node triangles`, equal only when no triad side has more than one connecting triple.
- **Clustering coefficient**: Global transitivity, `3 * node_triangles / wedges` where $$\text{wedges} = Σ_v C(\text{deg}_U(v), 2)$$ In other workds, `networkx.transitivity(U)`. The denominator ("wedges") is entirely determined by `U`'s degree distribution.

In [ ]:
from collections import Counter
from itertools import combinations
from pathlib import Path

import networkx as nx
import numpy as np
from rdflib import Graph


def load_triples(path):
    """Load (subject, predicate, object) triples from any graph file.

    Supports every RDF serialization rdflib can parse (`.ttl`, `.nt`,
    `.rdf`/`.xml`, ...) as well as plain delimited triples files (`.tsv`,
    `.csv`).
    """
    path = Path(path)
    if path.suffix.lower() in {".tsv", ".csv"}:
        sep = "\t" if path.suffix.lower() == ".tsv" else ","
        triples = []
        with open(path) as f:
            for line in f:
                line = line.rstrip("\n")
                if not line:
                    continue
                s, p, o = line.split(sep)
                triples.append((s, p, o))
        return triples

    g = Graph()
    g.parse(path)
    return [(str(s), str(p), str(o)) for s, p, o in g]


def graph_metrics(path):
    """Structural metrics for a single graph file."""
    triples = load_triples(path)

    nodes = {s for s, _, _ in triples} | {o for _, _, o in triples}

    # raw (multi-relational, directed) degree - every triple counts
    out_deg = Counter(s for s, _, _ in triples)
    in_deg = Counter(o for _, _, o in triples)
    out_degrees = np.array([out_deg[n] for n in nodes])
    in_degrees = np.array([in_deg[n] for n in nodes])

    # simple undirected projection: 1 edge per connected pair, dedup'd across
    # relation label and direction
    U = nx.Graph()
    U.add_nodes_from(nodes)
    mult = Counter()
    for s, _, o in triples:
        pair = frozenset((s, o))
        mult[pair] += 1
        U.add_edge(s, o)

    node_triangles = sum(nx.triangles(U).values()) // 3
    triads = [
        (u, v, w)
        for u, v, w in combinations(U.nodes(), 3)
        if U.has_edge(u, v) and U.has_edge(v, w) and U.has_edge(u, w)
    ]
    edge_triangles = sum(
        mult[frozenset((u, v))] * mult[frozenset((v, w))] * mult[frozenset((u, w))]
        for u, v, w in triads
    )

    return {
        "num_entities": len(nodes),
        "num_triples": len(triples),
        "num_edges": U.number_of_edges(),
        "num_node_triangles": node_triangles,
        "num_edge_triangles": edge_triangles,
        "max_in_degree": int(in_degrees.max()),
        "max_out_degree": int(out_degrees.max()),
        "std_in_degree": round(float(in_degrees.std()), 3),
        "std_out_degree": round(float(out_degrees.std()), 3),
        "clustering_coefficient": round(nx.transitivity(U), 3),
    }

#### Comparison between real and synthetic (PyGraft's) graph

`compare_graphs` below takes the paths to any two graph files (RDF or plain `.tsv`/`.csv` triples) and returns a side-by-side metrics table with the delta between them.

In [22]:
import pandas as pd


def compare_graphs(target_path, synthetic_path, target_label=None, synthetic_label=None):
    """Side-by-side graph_metrics for a target graph and a synthetic one,
    plus their delta.

    target_path, synthetic_path: any path load_triples/graph_metrics accepts
        (.ttl/.nt/.rdf or .tsv/.csv triples).
    target_label, synthetic_label: column names for the result; default to
        the file names.
    """
    target_label = target_label or f"Target ({Path(target_path).name})"
    synthetic_label = synthetic_label or f"Synthetic ({Path(synthetic_path).name})"

    metrics = {
        target_label: graph_metrics(target_path),
        synthetic_label: graph_metrics(synthetic_path),
    }
    df = pd.DataFrame(metrics)
    df["Δ (synthetic − target)"] = df[synthetic_label] - df[target_label]
    return df


df = compare_graphs(".data/mario/mario.tsv", ".data/mario/pygraft.tsv")
df

,Target (mario.tsv),Synthetic (pygraft.tsv),Δ (synthetic − target)
num_entities,15.000,15.000,0.000
num_triples,121.000,133.000,12.000
num_edges,65.000,88.000,23.000
num_node_triangles,148.000,270.000,122.000
num_edge_triangles,1112.000,912.000,-200.000
max_in_degree,14.000,14.000,0.000
max_out_degree,16.000,15.000,-1.000
std_in_degree,4.312,2.895,-1.417
std_out_degree,4.328,3.685,-0.643
clustering_coefficient,0.758,0.844,0.086


Both graphs have the same number of entities. The synthetic graph has slightly more triples spread over more distinct entity pairs, so on average each connected pair carries fewer parallel relations than in the real graph. This is visible in its lower multiplicity-weighted edge-triangle count, even though it has more node-level triangles and a higher clustering coefficient. In other words, the synthetic graph's local neighborhoods close into more triangles overall, but those triangles are built from more distinct edges rather than a few entity pairs linked by many relations at once, the way the real graph's denser pairs are.

The real graph's standard deviation of in-/out-degree (4.3 for both) is noticeably higher than the synthetic graph's (2.9 / 3.7), meaning the real graph's degrees are more skewed, while the synthetic graph's degrees are spread more evenly across its entities. This kind of degree-hub skew is typical of real-world or human-curated graphs, and it's exactly the sort of structure a generic synthetic-graph generator tends to under-reproduce. Constraining how evenly triples spread across *relations* says nothing about how evenly they spread across *entities*.

## Global metrics:

To measure global descriptors of the graph we use AMIE3 to mine rules from it. The rules describe the patterns within the relations that take place in the graph, rather than the graph topological features.

Each `.csv` here is AMIE3's rule output for one graph, produced with `run_amie.py`. A rule like `?a servantOf ?b => ?a allyOf ?b` doesn't reference any particular entity, only relation names. Since both graphs share the same schema, the rule *patterns* mined from the target and from the synthetic graph are directly comparable, without needing any entity alignment between the two graphs.

Two occurrences of the same pattern mined from different graphs aren't always spelled identically. `canonical_rule` below renames each rule's variables by order of first appearance, so equivalent patterns compare equal regardless.

`compare_rules` then outer-joins the two rule sets on this canonical pattern and reports:
- a `summary`: number of rules, shared or unique patterns, and the Jaccard similarity of the two rule sets;
- a `rules` table: one row per distinct pattern found in *either* graph, with each side's head coverage / confidence / support next to each other .

Both `load_rules` and `compare_rules` take an `exclude_predicates` argument: any rule that uses one of those predicates anywhere in its body or head is dropped before comparison. 
>`type` rules dominate AMIE's output (most mined rules are just "neighbor's type ⇒ own type"), so passing `exclude_predicates={"type"}` is a quick way to zoom in on the rules that relate the KG's "real" relations to each other instead.

In [3]:
def canonical_rule(rule_str):
    """Rename an AMIE rule's variables (`?a`, `?b`, ...) by order of first
    appearance, so the same relational pattern mined from different graphs
    compares equal regardless of AMIE's internal variable naming."""
    mapping = {}
    tokens = []
    for tok in rule_str.split():
        if tok.startswith("?"):
            tok = mapping.setdefault(tok, f"?v{len(mapping)}")
        tokens.append(tok)
    return " ".join(tokens)


def rule_predicates(rule_str):
    """The set of relation/predicate names used anywhere in an AMIE rule
    (body and head), e.g. {"enemyOf", "type"} for
    "?a enemyOf ?f ?f type ?b => ?a type ?b"."""
    return {tok for tok in rule_str.replace("=>", " ").split() if not tok.startswith("?")}


def load_rules(path, exclude_predicates=None):
    """Load an AMIE rules CSV (see mine_rules.py) indexed by canonical
    pattern, alongside its original columns.

    exclude_predicates: optional iterable of predicate/relation names (e.g.
        {"type"}) -- any mined rule using one of these predicates, anywhere
        in its body or head, is dropped.
    """
    df = pd.read_csv(path)
    if exclude_predicates:
        exclude_predicates = set(exclude_predicates)
        df = df[~df["rule"].map(lambda r: bool(rule_predicates(r) & exclude_predicates))]
    df = df.copy()
    df["canonical_rule"] = df["rule"].map(canonical_rule)
    return df.set_index("canonical_rule")


def compare_rules(
    target_path,
    synthetic_path,
    target_label=None,
    synthetic_label=None,
    exclude_predicates=None,
):
    """Compare AMIE-mined rules from a target graph and a synthetic one.

    target_path, synthetic_path: paths to AMIE rule CSVs (mine_rules.py output).
    target_label, synthetic_label: column labels; default to the file names.
    exclude_predicates: optional iterable of predicate names to drop rules
        for (forwarded to load_rules), e.g. {"type"}.

    Returns (summary, rules) -- see markdown above.
    """
    target_label = target_label or f"Target ({Path(target_path).name})"
    synthetic_label = synthetic_label or f"Synthetic ({Path(synthetic_path).name})"

    t = load_rules(target_path, exclude_predicates=exclude_predicates)
    s = load_rules(synthetic_path, exclude_predicates=exclude_predicates)
    t_ids, s_ids = set(t.index), set(s.index)

    metric_cols = ["head_coverage", "std_confidence", "pca_confidence", "positive_examples"]
    rules = pd.DataFrame(index=sorted(t_ids | s_ids))
    rules.index.name = "canonical_rule"
    rules["rule"] = [t["rule"][i] if i in t_ids else s["rule"][i] for i in rules.index]
    rules["in_target"] = rules.index.isin(t_ids)
    rules["in_synthetic"] = rules.index.isin(s_ids)
    for col in metric_cols:
        rules[f"{col} ({target_label})"] = [t.loc[i, col] if i in t_ids else np.nan for i in rules.index]
        rules[f"{col} ({synthetic_label})"] = [s.loc[i, col] if i in s_ids else np.nan for i in rules.index]
    rules = (
        rules.reset_index(drop=True)
        .set_index("rule")
        .sort_values(["in_target", "in_synthetic"], ascending=False)
    )

    summary = pd.Series(
        {
            "num_rules_target": len(t_ids),
            "num_rules_synthetic": len(s_ids),
            "num_shared": len(t_ids & s_ids),
            "num_target_only": len(t_ids - s_ids),
            "num_synthetic_only": len(s_ids - t_ids),
            "jaccard_similarity": round(len(t_ids & s_ids) / len(t_ids | s_ids), 3) if (t_ids | s_ids) else float("nan"),
        },
        name="value",
    )

    return summary, rules


rule_summary, rule_comparison = compare_rules(
    target_path=".data/mario/mario.csv",
    target_label="Real graph",
    synthetic_path=".data/mario/pygraft.csv",
    synthetic_label="PyGraft graph",
    exclude_predicates={"type"})
rule_summary

num_rules_target       3.0
num_rules_synthetic    0.0
num_shared             0.0
num_target_only        3.0
num_synthetic_only     0.0
jaccard_similarity     0.0
Name: value, dtype: float64

In [4]:
rule_comparison

,in_target,in_synthetic,head_coverage (Real graph),head_coverage (PyGraft graph),std_confidence (Real graph),std_confidence (PyGraft graph),pca_confidence (Real graph),pca_confidence (PyGraft graph),positive_examples (Real graph),positive_examples (PyGraft graph)
rule,,,,,,,,,,
?b allyOf ?a => ?a allyOf ?b,True,False,1.000000,NaN,1.0,NaN,1.0,NaN,52,NaN
?a servantOf ?b => ?a allyOf ?b,True,False,0.134615,NaN,1.0,NaN,1.0,NaN,7,NaN
?b servantOf ?a => ?a allyOf ?b,True,False,0.134615,NaN,1.0,NaN,1.0,NaN,7,NaN


With `type` rules excluded, mario.csv keeps exactly 3 rules flagged as target-only above and pygraft.csv has *zero* rules left. Once you factor out the "neighbor's type ⇒ own type" pattern both graphs share, PyGraft mines no relation-to-relation structure at all. Every bit of non-`type` relational logic AMIE recovers is specific to how mario.tsv was hand-authored, and PyGraft's schema-driven generation doesn't produce anything analogous.

- **Target-only** (3 rules): These rules encode actual logical structure in the real graph (a servant is always also counted as an ally) that PyGraft's synthetic graph does not reproduce.

While the two graphs' *topological* metrics track each other reasonably closely (previous section), the *relational* patterns AMIE recovers show a real structural gap: the real graph has a logical relationship between `allyOf` and `servantOf` that PyGraft's schema-driven generation doesn't capture.

